In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from keras.datasets import imdb
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

In [2]:
vocab_size = 10000
embedding_dim = 100
hidden_dim = 128
num_layers = 1
batch_size = 64
learning_rate = 0.001
epochs = 10
max_length = 256

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

print(len(X_train), "training samples")
print(len(X_test), "test samples")

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
25000 training samples
25000 test samples


In [4]:
from keras.preprocessing.sequence import pad_sequences

X_train = pad_sequences(X_train, maxlen=max_length)
X_test = pad_sequences(X_test, maxlen=max_length)

X_train = torch.tensor(X_train)
y_train = torch.tensor(y_train).float()

X_test = torch.tensor(X_test)
y_test = torch.tensor(y_test).float()

In [5]:
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [6]:
class SentimentModel(nn.Module):

    def __init__(self, rnn_type):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        if rnn_type == "RNN":
            self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers, batch_first=True)

        elif rnn_type == "LSTM":
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True)

        elif rnn_type == "GRU":
            self.rnn = nn.GRU(embedding_dim, hidden_dim, num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        x = self.embedding(x)

        output, _ = self.rnn(x)

        last_output = output[:, -1, :]

        out = self.fc(last_output)

        out = self.sigmoid(out)

        return out.squeeze()

In [7]:
def train_model(model):

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    model.to(device)

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for x, y in train_loader:

            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            outputs = model(x)

            loss = criterion(outputs, y)

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

In [8]:
def evaluate_model(model):

    model.eval()

    predictions = []
    true_labels = []

    with torch.no_grad():

        for x, y in test_loader:

            x = x.to(device)

            outputs = model(x)

            preds = (outputs.cpu() > 0.5).int()

            predictions.extend(preds.numpy())
            true_labels.extend(y.numpy())

    acc = accuracy_score(true_labels, predictions)
    precision = precision_score(true_labels, predictions)
    recall = recall_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions)
    cm = confusion_matrix(true_labels, predictions)

    print("Accuracy:", acc)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)
    print("Confusion Matrix:\n", cm)

In [9]:
print("Training RNN model")

rnn_model = SentimentModel("RNN")

train_model(rnn_model)

print("RNN Evaluation")

evaluate_model(rnn_model)

Training RNN model
Epoch 1 Loss: 258.8676
Epoch 2 Loss: 236.6837
Epoch 3 Loss: 200.2478
Epoch 4 Loss: 203.5506
Epoch 5 Loss: 179.3860
Epoch 6 Loss: 165.2500
Epoch 7 Loss: 166.6809
Epoch 8 Loss: 172.4720
Epoch 9 Loss: 166.9337
Epoch 10 Loss: 149.0746
RNN Evaluation
Accuracy: 0.71224
Precision: 0.6932828209237942
Recall: 0.76128
F1 Score: 0.7256920613132006
Confusion Matrix:
 [[8290 4210]
 [2984 9516]]


In [10]:
print("Training LSTM model")

lstm_model = SentimentModel("LSTM")

train_model(lstm_model)

print("LSTM Evaluation")

evaluate_model(lstm_model)

Training LSTM model
Epoch 1 Loss: 235.3606
Epoch 2 Loss: 198.6889
Epoch 3 Loss: 185.0471
Epoch 4 Loss: 140.2901
Epoch 5 Loss: 122.5462
Epoch 6 Loss: 104.3877
Epoch 7 Loss: 83.9627
Epoch 8 Loss: 71.6747
Epoch 9 Loss: 66.8238
Epoch 10 Loss: 86.8305
LSTM Evaluation
Accuracy: 0.75024
Precision: 0.724390243902439
Recall: 0.80784
F1 Score: 0.7638426626323752
Confusion Matrix:
 [[ 8658  3842]
 [ 2402 10098]]


In [11]:
print("Training GRU model")

gru_model = SentimentModel("GRU")

train_model(gru_model)

print("GRU Evaluation")

evaluate_model(gru_model)

Training GRU model
Epoch 1 Loss: 230.3580
Epoch 2 Loss: 157.7081
Epoch 3 Loss: 104.5214
Epoch 4 Loss: 78.7494
Epoch 5 Loss: 57.0259
Epoch 6 Loss: 40.8498
Epoch 7 Loss: 24.9956
Epoch 8 Loss: 14.4168
Epoch 9 Loss: 7.0040
Epoch 10 Loss: 6.2670
GRU Evaluation
Accuracy: 0.86972
Precision: 0.8932187526588956
Recall: 0.83984
F1 Score: 0.8657073351750298
Confusion Matrix:
 [[11245  1255]
 [ 2002 10498]]
